# Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/chendwend/thesis-assyrian-relief.git"
REPO_DIR = Path("/content/thesis-assyrian-relief")

# Google Drive locations
DRIVE_ROOT = Path("/content/drive/MyDrive/Graduate_Studies/Thesis")
IMAGE_ROOT = Path("/content/dataset")
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

print("IMAGE_ROOT:", IMAGE_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

IMAGE_ROOT: /content/drive/MyDrive/Graduate_Studies/Thesis/Dataset/dataset
OUTPUT_ROOT: /content/drive/MyDrive/Graduate_Studies/Thesis/outputs


In [ ]:
!unzip dataset.zip -d .
!rm -rf dataset.zip

In [ ]:
import os
import subprocess

if REPO_DIR.exists():
    %cd /content/thesis-assyrian-relief
    !git pull
else:
    %cd /content
    !git clone {REPO_URL}

%cd /content/thesis-assyrian-relief
!git status

In [ ]:
!pip install -q uv
!uv sync

In [ ]:
import yaml
from pathlib import Path

base_cfg_path = Path("configs/style_dinov2.yaml")
colab_cfg_path = Path("configs/style_dinov2_colab.yaml")

with open(base_cfg_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["data"]["csv_path"] = "data/splits/image_level_dataset.csv"
cfg["data"]["image_root"] = str(IMAGE_ROOT)
cfg["data"]["filename_sep"] = "-"

cfg["outputs"]["checkpoint_path"] = str(OUTPUT_ROOT / "checkpoints" / "dinov2_probe.pt")
cfg["outputs"]["history_path"] = str(OUTPUT_ROOT / "checkpoints" / "dinov2_probe.history.csv")
cfg["outputs"]["eval_metrics_path"] = str(OUTPUT_ROOT / "eval" / "test_metrics.json")
cfg["outputs"]["eval_retrieval_path"] = str(OUTPUT_ROOT / "eval" / "test_retrieval.csv")
cfg["outputs"]["umap_html_path"] = str(OUTPUT_ROOT / "umap" / "test_umap.html")
cfg["outputs"]["umap_csv_path"] = str(OUTPUT_ROOT / "umap" / "test_umap.csv")
cfg["outputs"]["retrieval_metrics_path"] = str(OUTPUT_ROOT / "retrieval" / "test_metrics.json")
cfg["outputs"]["retrieval_top1_path"] = str(OUTPUT_ROOT / "retrieval" / "test_top1.csv")
cfg["outputs"]["retrieval_topk_path"] = str(OUTPUT_ROOT / "retrieval" / "test_topk.csv")
cfg["outputs"]["retrieval_failures_path"] = str(OUTPUT_ROOT / "retrieval" / "test_failures.csv")

with open(colab_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print(f"Saved Colab config to: {colab_cfg_path}")
print(yaml.safe_dump(cfg, sort_keys=False))

## test

In [ ]:
!uv run python scripts/train_style.py \
  --config configs/style_dinov2_colab.yaml \
  --num-epochs 1 \
  --batch-size 8 \
  --num-workers 2

# Training

In [ ]:
!uv run python scripts/train_style.py \
  --config configs/style_dinov2_colab.yaml

# Evaluation

In [ ]:
!uv run python scripts/eval_style.py \
  --config configs/style_dinov2_colab.yaml \
  --eval-split test

# UMAP

In [ ]:
!uv run python scripts/umap_style.py \
  --config configs/style_dinov2_colab.yaml \
  --eval-split test

# Retrieval Analysis

In [ ]:
!uv run python scripts/retrieval_analysis.py \
  --config configs/style_dinov2_colab.yaml \
  --eval-split test